# Phase 1 — Step 1: Dataset Load & EDA
**Project:** Elder Abuse Detection & Legal Assistance — CSE 499A, NSU Spring 2026  
**Dataset:** `data/Elder_abuse_Dataset.csv` (199 cases)

---

## Cell 1 — Libraries Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LinearSegmentedColormap
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'font.size':          11,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'figure.facecolor':   'white',
    'axes.facecolor':     'white',
})

OUT_DIR = '../data'
DPI     = 180

RICH_PALETTE = [
    '#E63946','#2A9D8F','#E9C46A','#264653','#F4A261',
    '#A8DADC','#457B9D','#1D3557','#6D6875','#B5838D',
    '#E76F51','#52B788','#90E0EF','#C77DFF','#F72585',
    '#4CC9F0','#FF9F1C','#CBFF8C','#8338EC','#FB5607',
]
def make_colors(n):
    return RICH_PALETTE[:n] if n <= len(RICH_PALETTE) else (RICH_PALETTE * 3)[:n]

def save_chart(name):
    plt.savefig(f'{OUT_DIR}/{name}.png', dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[OK] {name}.png saved')

print('Libraries loaded')

## Cell 2 — Dataset Load

In [ ]:
df = pd.read_csv('../data/Elder_abuse_Dataset.csv', encoding='utf-8')

# Strip all string columns to avoid trailing-space issues
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print(f'Total rows    : {len(df)}')
print(f'Total columns : {len(df.columns)}')
print()
df.head()

## Cell 3 — Column Names

In [ ]:
print('All Columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

## Cell 4 — Data Types & Basic Info

In [ ]:
df.info()

## Cell 5 — Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if missing_df.empty:
    print('No missing values found')
else:
    print('Columns with missing values:')
    print(missing_df.to_string())

## Cell 6 — Normalize Columns (Category, Relation, Age, Date, Severity)

In [ ]:
# ── Abuse Category normalize ──────────────────────────────────────
cat_map = {
    'Physical':                                         'Physical Abuse',
    'Physcial':                                         'Physical Abuse',
    'Physical Abuse':                                   'Physical Abuse',
    'Physical and neglect':                             'Physical Abuse',
    'Verbally':                                         'Verbal Abuse',
    'verbally':                                         'Verbal Abuse',
    'Abandonment and physical':                         'Physical + Abandonment',
    'Neglect and Abandonment':                          'Neglect + Abandonment',
    'Financial Exploitation and Abandonment':           'Financial + Abandonment',
    'Financial Exploitation and physical':              'Financial + Physical',
    'Financial Exploitation and neglect':               'Financial + Neglect',
    'Financial Exploitation and Murder':                'Financial + Murder',
    'Financial Exploitation and Neglect':               'Financial + Neglect',
    'Financial Exploitation, physical and abandonment': 'Financial + Physical + Abandonment',
}
df['Category_Norm'] = df['Abuse Category'].replace(cat_map).fillna(df['Abuse Category'])

# ── Abuse Relation normalize ──────────────────────────────────────
rel_map = {
    'Son and daughter-in-law':          'Son & Daughter-in-law',
    'Son':                              'Son',
    'Children':                         'Children',
    'Daughter-in-law':                  'Daughter-in-law',
    'Family':                           'Family',
    'Neighbor':                         'Neighbor',
    'Relative':                         'Relative',
    'Neighbors':                        'Neighbor',
    'Homemaid':                         'House Help',
    'Landlord':                         'Landlord',
    'Grand son':                        'Grandchild',
    'Wife and Children':                'Spouse & Children',
    'Nephew and his wife':              'Nephew & Wife',
    'Husband and daughter-in-law':      'Husband & D-in-law',
    'Son-in-law and Grand son':         'Son-in-law & Grandson',
    'Son, daughter-in-law and grand son': 'Son & Family',
    'Son and Grand son':                'Son & Grandchild',
    'Union Council Member':             'Govt. Official',
    'Police':                           'Govt. Official',
}
df['Relation_Norm'] = df['Abuse Relation'].replace(rel_map).fillna(df['Abuse Relation'])

# ── Age numeric ───────────────────────────────────────────────────
df['Age_Numeric'] = pd.to_numeric(df['Age'], errors='coerce')

# ── Date & Year ───────────────────────────────────────────────────
df['Date_Parsed'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)
df['Year'] = df['Date_Parsed'].dt.year

# ── Severity Score (1-5) ──────────────────────────────────────────
severity_map = {
    'Murder': 5, 'Financial + Murder': 5,
    'Physical Abuse': 4, 'Financial + Physical': 4,
    'Physical + Abandonment': 4, 'Financial + Physical + Abandonment': 4,
    'Abandonment': 3, 'Neglect + Abandonment': 3, 'Financial + Abandonment': 3,
    'Financial Exploitation': 2, 'Neglect': 2, 'Financial + Neglect': 2,
    'Verbal Abuse': 1,
}
df['Severity_Score'] = df['Category_Norm'].map(severity_map).fillna(2)

# ── Trust Blind Spot ──────────────────────────────────────────────
family_kw = ['son','daughter','spouse','husband','wife','child','nephew','grandchild']
df['Trust_Blind_Spot'] = df['Relation_Norm'].str.lower().str.contains(
    '|'.join(family_kw), na=False).astype(int)

print(f'Category_Norm unique  : {df["Category_Norm"].nunique()}')
print(f'Relation_Norm unique  : {df["Relation_Norm"].nunique()}')
print(f'Age valid             : {df["Age_Numeric"].notna().sum()} / {len(df)}')
print(f'Trust Blind Spot      : {df["Trust_Blind_Spot"].sum()} cases ({df["Trust_Blind_Spot"].mean()*100:.1f}%)')
print(f'Severity Score range  : {int(df["Severity_Score"].min())} – {int(df["Severity_Score"].max())}')

## Cell 7 — Abuse Category (Raw counts)

In [ ]:
print('Abuse Category (Normalized):')
print(df['Category_Norm'].value_counts().to_string())

## Cell 8 — Abuse Relation (cleaned)

In [ ]:
print('Abuse Relation (Normalized):')
print(df['Relation_Norm'].value_counts().to_string())

## Cell 9 — Gender, Source, Age, Location

In [ ]:
print('--- Gender ---')
print(df['Gender'].value_counts().to_string())
print()
print('--- Data Type ---')
print(df['Data Type'].value_counts().to_string())
print()
print('--- Age Stats ---')
print(f'  Valid: {df["Age_Numeric"].notna().sum()}  |  Min: {df["Age_Numeric"].min():.0f}  |  Max: {df["Age_Numeric"].max():.0f}  |  Mean: {df["Age_Numeric"].mean():.1f}')
print()
print('--- Top 15 Locations ---')
loc_c = df['Location'].value_counts()
loc_c = loc_c[loc_c.index.str.lower() != 'unknown'].head(15)
print(loc_c.to_string())

---
## Charts (11 total)
All charts auto-saved to `data/` folder.

## Chart 1 — Abuse Category Distribution

In [ ]:
counts = df['Category_Norm'].value_counts()
colors = make_colors(len(counts))

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(counts.index[::-1], counts.values[::-1],
               color=colors[::-1], height=0.65, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, counts.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', ha='left', fontsize=10,
            fontweight='bold', color='#333333')

ax.set_title('Abuse Category Distribution', fontsize=16, fontweight='bold', pad=18, color='#1a1a2e')
ax.set_xlabel('Number of Cases', fontsize=12, color='#444')
ax.set_xlim(0, counts.max() + 8)
ax.tick_params(axis='y', labelsize=10)
ax.grid(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
plt.tight_layout()
save_chart('chart_01_abuse_category')

## Chart 2 — Abuser Relation Distribution

In [ ]:
rel_counts = df['Relation_Norm'].value_counts().head(12)
colors2    = make_colors(len(rel_counts))

fig, ax = plt.subplots(figsize=(12, 7))
bars2 = ax.barh(rel_counts.index[::-1], rel_counts.values[::-1],
                color=colors2[::-1], height=0.62, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars2, rel_counts.values[::-1]):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
            f"{val}  ({val/len(df)*100:.1f}%)",
            va='center', ha='left', fontsize=10, fontweight='bold', color='#333333')

ax.set_title('Who is the Abuser? — Relation Distribution', fontsize=16,
             fontweight='bold', pad=18, color='#1a1a2e')
ax.set_xlabel('Number of Cases', fontsize=12, color='#444')
ax.set_xlim(0, rel_counts.max() + 22)
ax.tick_params(axis='y', labelsize=10)
ax.grid(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
plt.tight_layout()
save_chart('chart_02_abuser_relation')

## Chart 3 — Top 15 Locations

In [ ]:
loc_counts = df['Location'].value_counts()
loc_counts = loc_counts[loc_counts.index.str.lower() != 'unknown'].head(15)
colors3    = make_colors(len(loc_counts))

fig, ax = plt.subplots(figsize=(12, 7))
bars3 = ax.bar(range(len(loc_counts)), loc_counts.values,
               color=colors3, edgecolor='white', linewidth=0.8, width=0.7)

for bar, val in zip(bars3, loc_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontsize=9.5,
            fontweight='bold', color='#333333')

ax.set_xticks(range(len(loc_counts)))
ax.set_xticklabels(loc_counts.index, rotation=38, ha='right', fontsize=10)
ax.set_title('Top 15 Locations by Number of Cases', fontsize=16,
             fontweight='bold', pad=18, color='#1a1a2e')
ax.set_ylabel('Number of Cases', fontsize=12, color='#444')
ax.set_ylim(0, loc_counts.max() + 5)
ax.grid(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
plt.tight_layout()
save_chart('chart_03_top15_locations')

## Chart 4 — Gender Distribution (Pie)

In [ ]:
gen_counts = df['Gender'].value_counts()
pie_colors = ['#457B9D','#E63946','#2A9D8F','#E9C46A'][:len(gen_counts)]

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    gen_counts.values, labels=None, autopct='%1.1f%%',
    colors=pie_colors, startangle=140, pctdistance=0.75,
    wedgeprops={'edgecolor':'white','linewidth':2.5},
    explode=[0.04]*len(gen_counts))
for t in autotexts:
    t.set_fontsize(13); t.set_fontweight('bold'); t.set_color('white')

legend_labels = [f"{g}  ({v})" for g, v in zip(gen_counts.index, gen_counts.values)]
ax.legend(wedges, legend_labels, title='Gender', title_fontsize=12,
          fontsize=11, loc='lower center', bbox_to_anchor=(0.5,-0.08), ncol=2, frameon=False)
ax.set_title('Victim Gender Distribution', fontsize=16, fontweight='bold', pad=20, color='#1a1a2e')
plt.tight_layout()
save_chart('chart_04_gender_pie')

## Chart 5 — Severity Score Distribution

In [ ]:
sev_labels = {1:'Verbal\n(Score 1)', 2:'Financial/Neglect\n(Score 2)',
              3:'Abandonment\n(Score 3)', 4:'Physical\n(Score 4)', 5:'Murder\n(Score 5)'}
sev_counts = df['Severity_Score'].value_counts().sort_index()
sev_colors = ['#52B788','#E9C46A','#F4A261','#E76F51','#E63946']

fig, ax = plt.subplots(figsize=(10, 6))
bars5 = ax.bar([sev_labels[s] for s in sev_counts.index],
               sev_counts.values,
               color=[sev_colors[int(i)-1] for i in sev_counts.index],
               edgecolor='white', linewidth=1.2, width=0.6)

for bar, val in zip(bars5, sev_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f"{val}\n({val/len(df)*100:.1f}%)",
            ha='center', va='bottom', fontsize=10.5, fontweight='bold', color='#333333')

ax.set_title('Severity Score Distribution', fontsize=16, fontweight='bold', pad=18, color='#1a1a2e')
ax.set_ylabel('Number of Cases', fontsize=12, color='#444')
ax.set_ylim(0, sev_counts.max()+12)
ax.grid(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
plt.tight_layout()
save_chart('chart_05_severity_score')

## Chart 6 — Age Distribution

In [ ]:
age_data = df['Age_Numeric'].dropna()

fig, ax = plt.subplots(figsize=(10, 6))
n, bins, patches = ax.hist(age_data, bins=12, edgecolor='white', linewidth=1.2, color='#457B9D', alpha=0.9)

cmap = plt.cm.Blues
for patch, val in zip(patches, n):
    patch.set_facecolor(cmap(0.35 + 0.55 * val / n.max()))

ax.axvline(age_data.mean(),   color='#E63946', linestyle='--', linewidth=2, label=f'Mean: {age_data.mean():.1f} yrs')
ax.axvline(age_data.median(), color='#2A9D8F', linestyle='--', linewidth=2, label=f'Median: {age_data.median():.1f} yrs')

ax.set_title('Victim Age Distribution', fontsize=16, fontweight='bold', pad=18, color='#1a1a2e')
ax.set_xlabel('Age (years)', fontsize=12, color='#444')
ax.set_ylabel('Number of Cases', fontsize=12, color='#444')
ax.legend(fontsize=11, frameon=False)
ax.grid(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
plt.tight_layout()
save_chart('chart_06_age_distribution')

## Chart 7 — Year-wise Case Trend

In [ ]:
year_counts = df['Year'].dropna().astype(int).value_counts().sort_index()
year_counts = year_counts[year_counts.index >= 2010]

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(year_counts.index, year_counts.values, alpha=0.18, color='#457B9D')
ax.plot(year_counts.index, year_counts.values, marker='o', markersize=8,
        color='#E63946', linewidth=2.5, markerfacecolor='white',
        markeredgecolor='#E63946', markeredgewidth=2.5)

for x, y in zip(year_counts.index, year_counts.values):
    ax.text(x, y+0.4, str(y), ha='center', va='bottom',
            fontsize=10, fontweight='bold', color='#333333')

ax.set_title('Year-wise Case Trend (2010–2026)', fontsize=16, fontweight='bold', pad=18, color='#1a1a2e')
ax.set_xlabel('Year', fontsize=12, color='#444')
ax.set_ylabel('Number of Cases', fontsize=12, color='#444')
ax.set_xticks(year_counts.index)
ax.tick_params(axis='x', rotation=45)
ax.set_ylim(0, year_counts.max()+4)
ax.grid(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
plt.tight_layout()
save_chart('chart_07_year_trend')

## Chart 8 — Heatmap: Abuse Category × Abuser Relation

In [ ]:
top_cats = df['Category_Norm'].value_counts().head(8).index.tolist()
top_rels = df['Relation_Norm'].value_counts().head(8).index.tolist()

heat_df = df[df['Category_Norm'].isin(top_cats) & df['Relation_Norm'].isin(top_rels)]
pivot   = heat_df.pivot_table(index='Category_Norm', columns='Relation_Norm',
                               aggfunc='size', fill_value=0)
pivot   = pivot.loc[[c for c in top_cats if c in pivot.index]]
pivot   = pivot[[c for c in top_rels if c in pivot.columns]]

cmap_heat = LinearSegmentedColormap.from_list('heat', ['#EAF4FB','#457B9D','#1D3557'])
fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(pivot.values, aspect='auto', cmap=cmap_heat)

ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns, rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(pivot.index, fontsize=10)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if val > 0:
            text_color = 'white' if val > pivot.values.max()*0.5 else '#1a1a2e'
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=11, fontweight='bold', color=text_color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Number of Cases', fontsize=11)
ax.set_title('Heatmap — Abuse Category x Abuser Relation', fontsize=16,
             fontweight='bold', pad=18, color='#1a1a2e')
plt.tight_layout()
save_chart('chart_08_heatmap_cat_relation')

## Chart 9 — Trust Blind Spot Analysis

In [ ]:
tbs_family = df['Trust_Blind_Spot'].sum()
tbs_other  = len(df) - tbs_family
tbs_labels = ['Family Member\n(Trust Blind Spot)', 'Non-Family\nAbuser']
tbs_vals   = [tbs_family, tbs_other]
tbs_colors = ['#E63946','#2A9D8F']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

wedges, texts, autos = ax1.pie(
    tbs_vals, labels=None, autopct='%1.1f%%', colors=tbs_colors, startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':3}, pctdistance=0.70, explode=[0.05,0])
for t in autos:
    t.set_fontsize(14); t.set_fontweight('bold'); t.set_color('white')
ax1.legend(wedges, [f"{l}  (n={v})" for l,v in zip(tbs_labels,tbs_vals)],
           fontsize=11, loc='lower center', bbox_to_anchor=(0.5,-0.12), frameon=False)
ax1.set_title('Trust Blind Spot Overview', fontsize=14, fontweight='bold', pad=15)

family_rels = df[df['Trust_Blind_Spot']==1]['Relation_Norm'].value_counts().head(8)
fam_colors  = make_colors(len(family_rels))
ax2.barh(family_rels.index[::-1], family_rels.values[::-1],
         color=fam_colors[::-1], height=0.6, edgecolor='white')
for i, val in enumerate(family_rels.values[::-1]):
    ax2.text(val+0.2, i, str(val), va='center', fontsize=10,
             fontweight='bold', color='#333333')
ax2.set_title('Family Abusers — Breakdown', fontsize=14, fontweight='bold', pad=15)
ax2.set_xlabel('Cases', fontsize=11)
ax2.set_xlim(0, family_rels.max()+10)
ax2.grid(False)
ax2.spines['left'].set_color('#cccccc')
ax2.spines['bottom'].set_color('#cccccc')

fig.suptitle(
    f'Trust Blind Spot — {tbs_family/len(df)*100:.1f}% of Abusers are Family Members',
    fontsize=15, fontweight='bold', color='#1a1a2e', y=1.01)
plt.tight_layout()
save_chart('chart_09_trust_blind_spot')

## Chart 10 — Data Source Type Distribution

In [ ]:
# Normalize source type (strip + merge variants)
df['Source_Norm'] = df['Data Type'].str.replace(r'[Pp]rimary\s*\(Interview\)', 'Primary (Interview)', regex=True)
df['Source_Norm'] = df['Source_Norm'].str.replace(r'[Ss]econdary\s*\(?Interview\)?', 'Secondary (Interview)', regex=True)
src_counts = df['Source_Norm'].value_counts()
src_colors = ['#457B9D','#2A9D8F','#E63946'][:len(src_counts)]

fig, ax = plt.subplots(figsize=(8, 8))
wedges2, texts2, autos2 = ax.pie(
    src_counts.values, labels=None, autopct='%1.1f%%',
    colors=src_colors, startangle=140,
    wedgeprops={'edgecolor':'white','linewidth':3},
    pctdistance=0.72, explode=[0.04]*len(src_counts))
for t in autos2:
    t.set_fontsize(13); t.set_fontweight('bold'); t.set_color('white')

ax.legend(wedges2, [f"{s}  (n={v})" for s,v in zip(src_counts.index, src_counts.values)],
          fontsize=11, loc='lower center', bbox_to_anchor=(0.5,-0.08), frameon=False)
ax.set_title('Data Source Type Distribution', fontsize=16, fontweight='bold', pad=20, color='#1a1a2e')
plt.tight_layout()
save_chart('chart_10_data_source_type')

## Chart 11 — Severity Score by Gender

In [ ]:
sev_gender = df.groupby(['Severity_Score','Gender']).size().unstack(fill_value=0)
sev_gender = sev_gender.reindex(sorted(sev_gender.index))

gender_colors = {'Male':'#457B9D','Female':'#E63946','Male & Female':'#2A9D8F'}
x    = np.arange(len(sev_gender.index))
cols = [c for c in sev_gender.columns if c in gender_colors]
w    = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
for i, col in enumerate(cols):
    offset = (i - len(cols)/2 + 0.5) * w
    bars_g = ax.bar(x+offset, sev_gender[col], w, label=col,
                    color=gender_colors[col], edgecolor='white', linewidth=0.8)
    for bar in bars_g:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x()+bar.get_width()/2, h+0.15,
                    str(int(h)), ha='center', va='bottom', fontsize=9, fontweight='bold')

sev_xlabels = {1:'Verbal\n(1)', 2:'Financial\n/Neglect (2)',
               3:'Abandonment\n(3)', 4:'Physical\n(4)', 5:'Murder\n(5)'}
ax.set_xticks(x)
ax.set_xticklabels([sev_xlabels.get(s,str(s)) for s in sev_gender.index], fontsize=10)
ax.set_title('Severity Score by Gender', fontsize=16, fontweight='bold', pad=18, color='#1a1a2e')
ax.set_ylabel('Number of Cases', fontsize=12, color='#444')
ax.legend(fontsize=11, frameon=False, loc='upper right')
ax.grid(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
plt.tight_layout()
save_chart('chart_11_severity_by_gender')

## Cell — Step 1 Summary

In [ ]:
print('=' * 58)
print('  STEP 1 COMPLETE — Dataset Load & EDA')
print('=' * 58)
print(f'  Total cases          : {len(df)}')
print(f'  Total columns (raw)  : 12')
print(f'  Category (raw)       : {df["Abuse Category"].nunique()} unique (with typos)')
print(f'  Category (normalized): {df["Category_Norm"].nunique()} unique')
print(f'  Relation (normalized): {df["Relation_Norm"].nunique()} unique')
print(f'  Age valid            : {df["Age_Numeric"].notna().sum()} / {len(df)}')
print(f'  Age range            : {int(df["Age_Numeric"].min())} – {int(df["Age_Numeric"].max())} yrs')
print(f'  Trust Blind Spot     : {df["Trust_Blind_Spot"].sum()} cases ({df["Trust_Blind_Spot"].mean()*100:.1f}%)')
print()
print('  Charts saved in data/ folder:')
charts = [
    'chart_01_abuse_category',
    'chart_02_abuser_relation',
    'chart_03_top15_locations',
    'chart_04_gender_pie',
    'chart_05_severity_score',
    'chart_06_age_distribution',
    'chart_07_year_trend',
    'chart_08_heatmap_cat_relation',
    'chart_09_trust_blind_spot',
    'chart_10_data_source_type',
    'chart_11_severity_by_gender',
]
for c in charts:
    print(f'  [OK] {c}.png')
print()
print('  NEXT -> Step 2: Dataset Cleaning + Train/Test Split')
print('=' * 58)